In [ ]:
# Notebook bootstrap: imports and core training utilities.
# Run this cell first.

# Libs Import
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)
import pandas as pd
import pickle as pkl
import os
import joblib
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.calibration import IsotonicRegression
from scipy.special import expit
import copy
from sklearn.model_selection import train_test_split

# device configuration (shared across training / evaluation)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# External validation on MIMIC (or another external cohort).
# Ensure features are aligned and the same scaler/calibrator are applied.

# ---- Input data paths (change to your local files) ----

UMN_TRAIN_CSV =  "data/umn/train.csv"
#"data/umn/master_dataset.csv"
UMN_TEST_CSV   = "data/umn/test.csv"
# Load different external datasets by changing these paths:
MIMIC_CSV      = "data//master_dataset.csv"
#"data/mimic/mimic_features_labels.csv"
STANFORD_CSV    = "data/stanford/stanford_features_labels.csv"

# ---- Output artifacts ----
OUTPUT_DIR = "output/"

MODEL_CKPT_PATH      = OUTPUT_DIR + "best_multidiseasepred.pt"
FEATURE_SCALER_PATH  = OUTPUT_DIR + "feature_scaler_umn.pkl"
CALIBRATOR_PATH      = OUTPUT_DIR + "calibrator_isotonic.pkl"
# ---- Reproducibility ----
SEED = 42


In [5]:
# Helper functions: data cleaning, metrics, bootstrapping, plotting.

# General Functions

def symmetric_kl(p, q, eps: float = 1e-8):
    """
    Symmetric KL divergence:
        0.5 * ( KL(p || q) + KL(q || p) )

    Args:
        p, q: probability distributions
              shape (..., E), last dim is expert dimension
        eps: numerical stability

    Returns:
        Tensor with shape (...)  (expert dimension reduced)
    """
    # avoid log(0)
    p = p.clamp_min(eps)
    q = q.clamp_min(eps)

    return 0.5 * (
        (p * (p.log() - q.log())).sum(dim=-1) +
        (q * (q.log() - p.log())).sum(dim=-1)
    )


def apply_expert_dropout(p, p_drop: float, training: bool):
    """
    Apply dropout on expert dimension (shared across all tasks & samples).

    This is expert-level dropout (NOT sample-level):
    - one mask per forward pass
    - shared by all tasks and all samples

    Args:
        p: expert distribution, shape (..., E)
        p_drop: dropout probability
        training: whether in training mode

    Returns:
        p_dropped: normalized expert distribution after dropout
        mask: dropout mask (for logging / analysis)
    """
    if (not training) or p_drop <= 0.0:
        return p, None

    # number of experts
    E = p.shape[-1]

    # inverted dropout mask: keep prob = 1 - p_drop
    mask = (torch.rand(E, device=p.device) > p_drop).float()   # [E]
    mask = mask / (1.0 - p_drop)

    # broadcast mask to match p's shape
    p = p * mask.view(*([1] * (p.dim() - 1)), E)

    # renormalize to probability simplex
    p = p / (p.sum(dim=-1, keepdim=True) + 1e-8)

    return p, mask

def ensure_feature_columns(
    df: pd.DataFrame,
    required_cols: list[str]
) -> pd.DataFrame:
    """
    Ensure that all required feature columns used during training exist
    (including columns such as `triage_pain`), and return the DataFrame
    with columns ordered exactly as in `required_cols`.

    Missing columns will be created and filled with NaN, so that downstream
    imputation (e.g., median / zero fill) can be applied consistently.
    """
    df = df.copy()

    # Identify missing feature columns
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        # Strategy A: restore missing features by creating empty columns
        # (placeholders filled with NaN; will be imputed later)
        for c in missing:
            df[c] = np.nan

        print(f"[WARN] Missing feature columns filled with NaN: {missing}")

    # Return features in the exact training-time order
    return df[required_cols]


# ----- Sensitivity / Specificity basic functions -----

def sensitivity_score(y_true, y_pred_bin):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_bin).ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def specificity_score(y_true, y_pred_bin):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_bin).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

def bootstrap_ci(metric_func, y_true, y_pred, n_bootstrap=2000, ci=0.95, seed=42):
    """Compute bootstrap-based confidence intervals for metrics."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    scores = []

    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        try:
            s = metric_func(y_true[idx], y_pred[idx])
            scores.append(s)
        except ValueError:
            continue

    if len(scores) == 0:
        return np.nan, (np.nan, np.nan)

    scores = np.array(scores)
    lower = np.percentile(scores, (1 - ci) / 2 * 100)
    upper = np.percentile(scores, (1 + ci) / 2 * 100)

    return np.mean(scores), (lower, upper)

def find_optimal_threshold(y_true, y_prob, strategy: str = "youden"):
    """
    Find the optimal classification threshold based on the ROC curve.

    strategy="youden":
        Maximize Youden's J statistic = sensitivity + specificity - 1
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    # Guard against degenerate cases (all 0 or all 1 labels)
    if len(thresholds) == 0 or np.all(np.isnan(tpr)) or np.all(np.isnan(fpr)):
        return 0.5, np.nan, np.nan

    sens = tpr              # True Positive Rate
    spec = 1.0 - fpr        # True Negative Rate

    if strategy == "youden":
        J = sens + spec - 1.0
        idx = np.nanargmax(J)
    else:
        # Other strategies can be extended here
        idx = np.nanargmax(sens)

    thr_opt = thresholds[idx]
    sens_opt = sens[idx]
    spec_opt = spec[idx]

    return float(thr_opt), float(sens_opt), float(spec_opt)

def bootstrap_sens_spec_ci(y_true, y_prob, n_boot: int = 1000, alpha: float = 0.95):
    """
    Bootstrap confidence intervals for sensitivity and specificity,
    while re-optimizing the threshold in each bootstrap sample.

    Returns:
        (sens_mean, (sens_low, sens_high)),
        (spec_mean, (spec_low, spec_high)),
        thr_mean
    """
    rng = np.random.default_rng(42)
    n = len(y_true)

    sens_list, spec_list, thr_list = [], [], []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_prob[idx]

        thr_b, sens_b, spec_b = find_optimal_threshold(
            yt, yp, strategy="youden"
        )

        sens_list.append(sens_b)
        spec_list.append(spec_b)
        thr_list.append(thr_b)

    sens_arr = np.array(sens_list)
    spec_arr = np.array(spec_list)
    thr_arr = np.array(thr_list)

    lower = (1.0 - alpha) / 2.0 * 100.0
    upper = (1.0 + alpha) / 2.0 * 100.0

    sens_mean = float(np.nanmean(sens_arr))
    sens_low = float(np.nanpercentile(sens_arr, lower))
    sens_high = float(np.nanpercentile(sens_arr, upper))

    spec_mean = float(np.nanmean(spec_arr))
    spec_low = float(np.nanpercentile(spec_arr, lower))
    spec_high = float(np.nanpercentile(spec_arr, upper))

    thr_mean = float(np.nanmean(thr_arr))

    return (
        (sens_mean, (sens_low, sens_high)),
        (spec_mean, (spec_low, spec_high)),
        thr_mean
    )

def macro_ci_over_tasks(vals, n_boot=5000, ci=0.95, seed=42):
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]

    if len(vals) == 0:
        return np.nan, (np.nan, np.nan)

    if len(vals) == 1:
        return float(vals[0]), (np.nan, np.nan)

    rng = np.random.default_rng(seed)
    n = len(vals)

    boot = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot.append(np.mean(vals[idx]))

    boot = np.asarray(boot, dtype=float)

    alpha = 1 - ci
    low = float(np.percentile(boot, 100 * (alpha / 2)))
    high = float(np.percentile(boot, 100 * (1 - alpha / 2)))

    return float(np.mean(vals)), (low, high)


In [6]:
# Model definition: DSelect-K style multi-task network.

# Model Structure

class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        # shared expert MLP
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        """
        Args:
            x: [B, input_dim]

        Returns:
            expert representation: [B, hidden_dim]
        """
        return self.net(x)
    
class Tower(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        # single logit per task (binary classification)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        """
        Args:
            x: [B, hidden_dim]

        Returns:
            logits: [B, 1]
        """
        return self.fc(x)

class DSelectKLayerReg(nn.Module):
    """
    DSelect-K layer with two regularizations:

    1) Shared-router regularization (symmetric KL)
       - aligns task-specific and shared routing

    2) Expert-dropout regularization
       - improves robustness by dropping experts
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_experts: int,
        num_tasks: int,
        k: int,
        temperature: float = 1.0,

        # regularization knobs
        alpha_mix_with_shared: float = 0.1,   # mixing ratio alpha
        lambda_shared_kl: float = 0.05,        # KL regularization weight
        expert_dropout_p: float = 0.0          # expert dropout probability
    ):
        super().__init__()

        self.num_experts = num_experts
        self.num_tasks = num_tasks
        self.k = k
        self.temperature = temperature
            # shared experts
        self.experts = nn.ModuleList(
            [Expert(input_dim, hidden_dim) for _ in range(num_experts)]
        )

        # task-specific, input-agnostic parameters
        # S: selection logits, A: attention logits
        # shape: [T, E]
        self.S = nn.Parameter(torch.zeros(num_tasks, num_experts))
        self.A = nn.Parameter(torch.zeros(num_tasks, num_experts))

        # shared router: input-dependent expert distribution
        self.shared_router = nn.Linear(input_dim, num_experts)

        # regularization hyperparameters
        self.alpha = alpha_mix_with_shared
        self.lambda_shared = lambda_shared_kl
        self.p_drop = expert_dropout_p

    def _task_distribution(self):
        """
        Compute task-specific expert distributions p_task.

        Returns:
            p_task: [T, E]
        """

        # selection distribution
        sel = self.S / self.temperature
        sel_w = F.softmax(sel, dim=-1)

        # attention distribution
        att = self.A / self.temperature
        att_w = F.softmax(att, dim=-1)

        # 50/50 combination (as in original design)
        p_task = 0.5 * sel_w + 0.5 * att_w

        # numerical renormalization
        p_task = p_task / (p_task.sum(dim=-1, keepdim=True) + 1e-8)

        return p_task

    def forward(self, x):
        """
        Args:
            x: [B, input_dim]

        Returns:
            reps: list of length T, each [B, hidden_dim]
            aux:
                - shared_router_reg: KL regularization term
                - expert_dropout_ratio: monitoring statistic
        """
        B = x.size(0)

        # 1) expert outputs: [B, E, H]
        expert_outs = torch.stack([e(x) for e in self.experts], dim=1)

        # 2) task-specific routing: [T, E]
        p_task = self._task_distribution()

        # 3) shared routing (input-dependent): [B, E]
        router_logits = self.shared_router(x)
        p_shared = F.softmax(router_logits / self.temperature, dim=-1)

        aux = {}

        # 4) expert dropout (training only)
        p_shared_drop, mask = apply_expert_dropout(
            p_shared, self.p_drop, self.training
        )
        if mask is not None:
            aux["expert_dropout_ratio"] = float((mask == 0).float().mean().item())

        # broadcast to [T, B, E]
        p_task_b = p_task.unsqueeze(1).expand(self.num_tasks, B, self.num_experts)
        p_shared_b = p_shared_drop.unsqueeze(0).expand(self.num_tasks, B, self.num_experts)

        # 5) shared-router KL regularization
        shared_reg = symmetric_kl(p_task_b, p_shared_b).mean()
        aux["shared_router_reg"] = self.lambda_shared * shared_reg

        # 6) mixture of task & shared routing
        p_mix = (1.0 - self.alpha) * p_task_b + self.alpha * p_shared_b
        p_mix = p_mix / (p_mix.sum(dim=-1, keepdim=True) + 1e-8)

        # 7) weighted sum of expert outputs
        reps = []
        for t in range(self.num_tasks):
            reps_t = torch.einsum("be,beh->bh", p_mix[t], expert_outs)
            reps.append(reps_t)

        return reps, aux


class DSelectKModelReg(nn.Module):
    """
    Multi-task model wrapper:
    - shared DSelect-K layer
    - one binary tower per task
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_experts: int,
        num_tasks: int,
        k: int,
        temperature: float = 1.0,
        alpha_mix_with_shared: float = 0.1,
        lambda_shared_kl: float = 0.05,
        expert_dropout_p: float = 0.0,
    ):
        super().__init__()

        self.num_tasks = num_tasks

        self.shared_layer = DSelectKLayerReg(
            input_dim, hidden_dim,
            num_experts, num_tasks, k,
            temperature,
            alpha_mix_with_shared=alpha_mix_with_shared,
            lambda_shared_kl=lambda_shared_kl,
            expert_dropout_p=expert_dropout_p,
        )

        # one tower per task
        self.towers = nn.ModuleList(
            [Tower(hidden_dim) for _ in range(num_tasks)]
        )

    def forward(self, x):
        """
        Args:
            x: [B, input_dim]

        Returns:
            logits: [B, T]
            aux: auxiliary regularization terms
        """
        reps, aux = self.shared_layer(x)

        logits = torch.cat(
            [self.towers[t](reps[t]) for t in range(self.num_tasks)],
            dim=1
        )

        return logits, aux

In [7]:
# Model definition: DSelect-K style multi-task network.

# Training Function

def run_epoch(
    dl,
    model,
    optimizer,
    criterion,
    phase: str = "train",
    return_preds: bool = False,
    l1_lambda: float = 0.0,
    aux_lambda: float = 1.0,
):
    """
    Run one epoch of training or validation.

    Args:
        dl          : DataLoader yielding (xb, yb)
                      xb: [batch, input_dim]
                      yb: [batch, num_tasks] (0/1 labels)
        model       : DSelectKModelReg
                      returns (logits, aux) OR legacy model (logits only)
        optimizer   : torch.optim.Optimizer
        criterion   : e.g. nn.BCEWithLogitsLoss(reduction="mean")
        phase       : "train" or "val"
        return_preds: if True, also return per-task preds & trues
        l1_lambda   : L1 regularization coefficient on gating params S/A
        aux_lambda  : weight for auxiliary regularization terms

    Returns:
        if return_preds:
            epoch_loss, aucs, prcs, preds, trues
        else:
            epoch_loss, aucs, prcs
    """
    is_train = (phase == "train")
    model.train() if is_train else model.eval()
    total_loss = 0.0
    num_tasks = model.num_tasks

    # collect per-task predictions / labels
    preds = [[] for _ in range(num_tasks)]
    trues = [[] for _ in range(num_tasks)]

    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)

        with torch.set_grad_enabled(is_train):
            # compatible with:
            #   - new model: (logits, aux)
            #   - legacy model: logits only
            out = model(xb)
            if isinstance(out, tuple):
                logits, aux = out
            else:
                logits, aux = out, {}
            # primary loss
            # logits: [batch, num_tasks]
            loss = criterion(logits, yb)
            # -------- L1 on gating params (your original logic) --------
            if is_train and l1_lambda > 0.0:
                # NOTE: path matches your model definition exactly
                l1_reg = (
                    model.shared_layer.S.abs().sum() +
                    model.shared_layer.A.abs().sum()
                )
                loss = loss + l1_lambda * l1_reg
            # -------- auxiliary regularizations (shared-router, etc.) --------
            if is_train and aux_lambda > 0.0 and isinstance(aux, dict):
                # currently your model returns:
                # aux["shared_router_reg"] : scalar tensor
                if "shared_router_reg" in aux and aux["shared_router_reg"] is not None:
                    loss = loss + aux_lambda * aux["shared_router_reg"]

                # extensible: if you later add entropy / diversity regs
                if "entropy_reg" in aux and aux["entropy_reg"] is not None:
                    loss = loss + aux_lambda * aux["entropy_reg"]
            if is_train:
                optimizer.zero_grad()
                loss.backward()

                # gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)

                optimizer.step()
        # -------- accumulate epoch stats / preds --------
        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size

        # sigmoid probabilities for metrics
        # We keep *raw logits* per task so that calibration (isotonic) can be fit/applied consistently on the same scale.
        # Metrics (AUROC/AUPRC) are still computed on sigmoid(logits) probabilities.
        logits_cpu = logits.detach().cpu()               # [batch, num_tasks]
        y_true = yb.detach().cpu()                       # [batch, num_tasks]

        for t in range(num_tasks):
            preds[t].append(logits_cpu[:, t])            # store logits (NOT probabilities)
            trues[t].append(y_true[:, t])
    epoch_loss = total_loss / len(dl.dataset)

    aucs, prcs = [], []

    for p_list, t_list in zip(preds, trues):
        p_logits = torch.cat(p_list)                     # logits, shape [N_task]
        p = torch.sigmoid(p_logits).numpy()              # probabilities for metrics
        t = torch.cat(t_list).numpy()                    # true labels

        try:
            aucs.append(roc_auc_score(t, p))
            prcs.append(average_precision_score(t, p))
        except ValueError:
            # happens when only one class is present
            aucs.append(np.nan)
            prcs.append(np.nan)
            
    if return_preds:
        return epoch_loss, aucs, prcs, preds, trues
    else:
        return epoch_loss, aucs, prcs

## Model Training

In [8]:
# Run this cell as part of the pipeline.

# Feature and outcome lists
col_list = [
    "age", "gender", "n_ed_30d", "n_ed_90d", "n_ed_365d", "n_hosp_30d", "n_hosp_90d", "n_hosp_365d", "n_icu_30d", "n_icu_90d", "n_icu_365d", "triage_temperature", "triage_heartrate", "triage_resprate", "triage_o2sat", "triage_sbp", "triage_dbp", "triage_acuity", "chiefcom_chest_pain", "chiefcom_abdominal_pain", "chiefcom_headache", "chiefcom_shortness_of_breath", "chiefcom_back_pain", "chiefcom_cough", "chiefcom_nausea_vomiting", "chiefcom_fever_chills", "chiefcom_syncope", "chiefcom_dizziness", "cci_MI", "cci_CHF", "cci_PVD", "cci_Stroke", "cci_Dementia", "cci_Pulmonary", "cci_Rheumatic", "cci_PUD", "cci_Liver1", "cci_DM1", "cci_DM2", "cci_Paralysis", "cci_Renal", "cci_Cancer1", "cci_Liver2", "cci_Cancer2", "cci_HIV", "eci_Arrhythmia", "eci_Valvular", "eci_PHTN",  "eci_HTN1", "eci_HTN2", "eci_NeuroOther", "eci_Hypothyroid", "eci_Lymphoma", "eci_Coagulopathy", "eci_Obesity", "eci_WeightLoss", "eci_FluidsLytes", "eci_BloodLoss", "eci_Anemia", "eci_Alcohol", "eci_Drugs","eci_Psychoses", "eci_Depression"
]
outcome_list = [
    "outcome_hospitalization", "outcome_critical", "outcome_sepsis", "outcome_pneumonia_viral", "outcome_ards", "outcome_pe", "outcome_copd_asthma", "outcome_acs_mi", "outcome_aki"
]

In [11]:
# NOTE: Replace any remaining absolute paths with PROJECT_ROOT-relative paths.
# Helper functions: data cleaning, metrics, bootstrapping, plotting.

# Training Dta Preparation

# ======== 1) Build feature & label DataFrame ========
df = pd.read_csv(str(UMN_TRAIN_CSV)) # Load your dataset here

raw_feats = ensure_feature_columns(df, col_list)

# ======== Explicitly encode gender: M -> 0, F -> 1 ========
if "gender" in raw_feats.columns:
    raw_feats["gender"] = (
        raw_feats["gender"]
        .astype(str)
        .str.upper()
        .map({"M": 0, "F": 1})
    )

# Keep only numeric / boolean features
feat_df = raw_feats.select_dtypes(include=[np.number, bool]).copy()

# Force numeric conversion (invalid entries -> NaN)
feat_df = feat_df.apply(pd.to_numeric, errors="coerce")

# Compute median values on the training data
feat_median = feat_df.median()

# Fill missing values using feature-wise medians
feat_df = feat_df.fillna(feat_median)

# Save feature column order (for test / external alignment)
feat_cols = feat_df.columns.tolist()


# ======== 2) StandardScaler ========

scaler = StandardScaler()
X_np = scaler.fit_transform(feat_df.to_numpy(dtype=float))

# Save scaler & preprocessing metadata for reuse
# (e.g., other notebooks or external validation)
joblib.dump(
    {
        "scaler": scaler,
        "median": feat_median,
        "feat_cols": feat_cols
    },
    str(FEATURE_SCALER_PATH)
)


# ======== 3) Process labels ========

y_np = df.loc[:, outcome_list].to_numpy(dtype=float)


# ======== 4) Convert to tensors & split train/val ========

X_t = torch.from_numpy(X_np).float()
y_t = torch.from_numpy(y_np).float()
dataset = TensorDataset(X_t, y_t)

n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val = n_total - n_train

train_ds, val_ds = random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

batch_size = 256

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_dl = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False
)

In [12]:
# Model definition: DSelect-K style multi-task network.

# Model Parameters Setup

input_dim = X_np.shape[1]

model = DSelectKModelReg(
    input_dim=input_dim,
    hidden_dim=256,
    num_experts=13,
    num_tasks=9,
    k=7,                         # compatibility parameter (hard top-k not enforced)
    temperature=0.2,
    alpha_mix_with_shared=0.1,   # mixing ratio with shared router
    lambda_shared_kl=0.05,       # symmetric KL regularization weight
    expert_dropout_p=0.1         # expert-level dropout probability
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

best_val_auc = -np.inf
num_epochs = 10

In [ ]:
# Run this cell as part of the pipeline.

# Training Loop

save_path = str(MODEL_CKPT_PATH)  # where to save best checkpoint

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch:02d}")

    # Run one training epoch
    tr_loss, tr_aucs, tr_prcs = run_epoch(
        train_dl,
        model,
        optimizer,
        criterion,
        "train",
        l1_lambda=1e-4,
        aux_lambda=1.0
    )

    # Run one validation epoch
    vl_loss, vl_aucs, vl_prcs = run_epoch(
        val_dl,
        model,
        optimizer,
        criterion,
        "val",
        l1_lambda=1e-4,
        aux_lambda=1.0
    )

    # Compute mean validation metrics across tasks (ignore NaNs)
    mean_val_auc = np.nanmean(vl_aucs)
    mean_val_prc = np.nanmean(vl_prcs)

    # Step LR scheduler based on validation AUROC
    scheduler.step(mean_val_auc)

    # Log epoch-level metrics
    print(
        f"Train loss {tr_loss:.4f} | Val loss {vl_loss:.4f} | "
        f"Val AUC {mean_val_auc:.4f} | Val PRC {mean_val_prc:.4f}"
    )

    # Save best model according to mean validation AUROC
    if mean_val_auc > best_val_auc:
        best_val_auc = mean_val_auc

        # Ensure target directory exists
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        # Save model parameters only (state_dict)
        torch.save(model.state_dict(), save_path)

        print(f"  Saved new best to {save_path}")

NameError: name 'num_epochs' is not defined

## Internal/External Validation with Model Calibration

In [ ]:
# Run this cell as part of the pipeline.

# Load external / test dataset
df_test = pd.read_csv(str(MIMIC_CSV)) # Change to your external dataset path

In [14]:
# Calibration: fit per-task isotonic regression on VALIDATION logits.
# IMPORTANT: fit and apply must use the SAME score scale (logits).

# Isotonic Calibrator

class IsotonicCalibrator:
    """
    Task-wise isotonic regression calibrator.
    """

    def __init__(self, num_tasks):
        self.num_tasks = num_tasks
        self.calibrators = [
            IsotonicRegression(out_of_bounds="clip")
            for _ in range(num_tasks)
        ]

    def fit(self, logits, targets):
        """
        Fit one calibrator per task.
        """
        for i in range(self.num_tasks):
            self.calibrators[i].fit(logits[:, i], targets[:, i])

    def predict(self, logits):
        """
        Apply calibration to logits.
        """
        calibrated_probs = np.zeros_like(logits)
        for i in range(self.num_tasks):
            calibrated_probs[:, i] = self.calibrators[i].predict(logits[:, i])
        return calibrated_probs

In [25]:
# External Validation Data Preparation Helpers

def split_external_val_test_by_group(
    df: pd.DataFrame,
    group_col: str,
    val_frac: float = 0.2,
    seed: int = 42,
    outcome_cols: list[str] | None = None,
    stratify_mode: str | None = "any_positive",
):
    """
    Generic external split by group (patient/encounter).
    - group_col: patient_id / subject_id / encounter_id...
    - stratify_mode:
        - None: random split
        - "any_positive": stratify by whether a group has any positive label across outcome_cols
    """
    df = df.copy()
    df = df[df[group_col].notna()]
    groups = df[group_col].unique()
    groups = np.array(groups)

    stratify_labels = None
    if stratify_mode == "any_positive":
        if outcome_cols is None or len(outcome_cols) == 0:
            raise ValueError("outcome_cols must be provided for stratify_mode='any_positive'")
        # aggregate group label: any positive across outcomes, any row
        grp_any_pos = (
            df.groupby(group_col)[outcome_cols]
              .max(numeric_only=True)  # assumes 0/1
              .max(axis=1)
              .reindex(groups)
              .fillna(0)
              .astype(int)
              .values
        )
        stratify_labels = grp_any_pos
        
        # if only one class present, fallback to no stratification
        if len(np.unique(stratify_labels)) < 2:
            stratify_labels = None

    g_val, g_test = train_test_split(
        groups,
        test_size=1.0 - val_frac,
        random_state=seed,
        shuffle=True,
        stratify=stratify_labels,
    )

    df_val = df[df[group_col].isin(g_val)].copy()
    df_test = df[df[group_col].isin(g_test)].copy()
    return df_val, df_test


def default_preprocess_fn(raw_feats: pd.DataFrame) -> pd.DataFrame:
    """No-op preprocess. Override per external dataset if needed."""
    return raw_feats

def preprocess_other(raw_feats: pd.DataFrame) -> pd.DataFrame:
    raw_feats = raw_feats.copy()
    if "gender" in raw_feats.columns:
        raw_feats["gender"] = (
            raw_feats["gender"].astype(str).str.lower()
            .map({"male": 0, "female": 1})
        )
    return raw_feats

def build_dataloader_from_df_generic(
    df: pd.DataFrame,
    label_cols: list[str],
    feature_scaler_path: str,
    batch_size: int = 256,
    preprocess_fn=preprocess_other,
    return_numpy: bool = False,
):
    """
    Build dataloader for any external dataset.
    Assumes df already has label_cols and id_col.
    Feature columns are aligned using saved feat_cols from feature_scaler_path.
    """
    saved = joblib.load(feature_scaler_path)
    scaler = saved["scaler"]
    feat_median = saved["median"]
    feat_cols = saved["feat_cols"]

    # 1) align features to training feat_cols (your existing helper)
    raw_feats = ensure_feature_columns(df, feat_cols)

    # 2) dataset-specific preprocess
    raw_feats = preprocess_fn(raw_feats)

    # 3) numeric casting + imputation with training medians
    feat_df = raw_feats.select_dtypes(include=[np.number, bool]).copy()
    feat_df = feat_df.apply(pd.to_numeric, errors="coerce")
    feat_df = feat_df.fillna(feat_median)

    # 4) scale
    X_np = scaler.transform(feat_df.to_numpy(dtype=float))
    y_np = df.loc[:, label_cols].to_numpy(dtype=float)

    # 5) torch tensors
    X_t = torch.from_numpy(X_np).float()
    y_t = torch.from_numpy(y_np).float()

    dataset = TensorDataset(X_t, y_t)
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    if return_numpy:
        return dl, X_np, y_np
    return dl



In [26]:
# ====== External dataset inputs ======
# Generate external val dataloaders to fit calibrator and test dataloaders for final evaluation.

df_external = df_test   # Any external dataset DataFrame

cfg = {
    "label_cols": outcome_list,      # outcome columns
    "id_col": "subject_id",        # group column for splitting
    "val_frac": 0.2,
    "seed": 42,
    "batch_size": 256,
    "feature_scaler_path": str(FEATURE_SCALER_PATH),
    "stratify_mode": "any_positive", # None or "any_positive"
    "preprocess_fn": preprocess_other,  # data preprocess function: default_preprocess_fn or preprocess_other
}

# 1) split
ext_val_df, ext_test_df = split_external_val_test_by_group(
    df_external,
    group_col=cfg["id_col"],
    val_frac=cfg["val_frac"],
    seed=cfg["seed"],
    outcome_cols=cfg["label_cols"],
    stratify_mode=cfg["stratify_mode"],
)

# 2) dataloaders
ext_val_dl = build_dataloader_from_df_generic(
    ext_val_df,
    label_cols=cfg["label_cols"],
    feature_scaler_path=cfg["feature_scaler_path"],
    batch_size=cfg["batch_size"],
    preprocess_fn=cfg["preprocess_fn"],
)

ext_test_dl = build_dataloader_from_df_generic(
    ext_test_df,
    label_cols=cfg["label_cols"],
    feature_scaler_path=cfg["feature_scaler_path"],
    batch_size=cfg["batch_size"],
    preprocess_fn=cfg["preprocess_fn"],
)

print("External val groups:", ext_val_df[cfg["id_col"]].nunique(), "rows:", len(ext_val_df))
print("External test groups:", ext_test_df[cfg["id_col"]].nunique(), "rows:", len(ext_test_df))


External val groups: 43342 rows: 90576
External test groups: 173372 rows: 358228


In [29]:
# Calibration: fit per-task isotonic regression on VALIDATION logits.
# IMPORTANT: fit and apply must use the SAME score scale (logits).

# Generarte Calibrator

calibrator_path = str(CALIBRATOR_PATH)

# ======== 1. If a pre-fitted calibrator exists, load it directly ========
if os.path.exists(calibrator_path):
    calibrator = joblib.load(calibrator_path)
    print("Loaded pre-fitted calibrator from file:", calibrator_path)

# ======== 2. Otherwise, fit a new calibrator using validation logits / targets ========
else:
    print("Calibrator file not found, fitting a new one ...")
    calibrator = {}

    model.eval()

    _, _, _, val_preds_list, val_trues_list = run_epoch(
        ext_val_dl,                 # external validation dataloader
        model,
        optimizer=None,
        criterion=criterion,
        phase="val",
        return_preds=True,
        l1_lambda=0.0,
        aux_lambda=0.0
    )

    # Convert list-of-lists to numpy arrays
    # Each element in val_preds_list / val_trues_list corresponds to one task

    val_logits = np.stack(
        [torch.cat(p).numpy() for p in val_preds_list],
        axis=1
    )   # shape: (N, T)

    val_targets = np.stack(
        [torch.cat(t).numpy() for t in val_trues_list],
    axis=1
    )   # shape: (N, T)
    # val_logits shape: (N, T)
    # val_targets shape: (N, T)
    for t_idx, task_name in enumerate(outcome_list):
        y_true = val_targets[:, t_idx]
        y_score = val_logits[:, t_idx]   # can use logits or sigmoid probabilities

        # Only perform calibration if both classes (0/1) are present
        if len(np.unique(y_true)) < 2:
            print(f"Task {task_name} only has one class, skip isotonic.")
            continue

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(y_score, y_true)
        calibrator[task_name] = iso

    # Save the newly fitted isotonic calibrator
    joblib.dump(calibrator, calibrator_path)
    print("Saved new isotonic calibrator to:", calibrator_path)

Calibrator file not found, fitting a new one ...
Saved new isotonic calibrator to: E:/UMN科研/Model_training/MultiDieasesPred/output/calibrator_isotonic.pkl


In [ ]:
# Calibration: fit per-task isotonic regression on VALIDATION logits.
# IMPORTANT: fit and apply must use the SAME score scale (logits).

# Evaluate the model on the test set with isotonic calibration

model.load_state_dict(
    torch.load(
        str(MODEL_CKPT_PATH),  # Change to your saved model path
        weights_only=True,
        map_location=torch.device("cpu")
    )
) # Load the best saved model weights (CPU mapping for portability)
model.eval()
test_logits = []
test_targets = []

with torch.no_grad():
    for X_batch, y_batch, pid_batch in ext_test_dl:
        X_batch = X_batch.to(device).float()
        y_batch = y_batch.to(device).float()

        outputs = model(X_batch)

        # ======== Unify outputs -> logits tensor with shape (batch, T) ========
        if isinstance(outputs, dict):
            # {task_name: tensor} -> concatenate in outcome_list order
            logits_tensor_list = [
                outputs[task].detach().cpu() for task in outcome_list
            ]
            logits = torch.cat(logits_tensor_list, dim=1)

        elif isinstance(outputs, (list, tuple)):
            # Possible cases:
            # 1) (logits, aux, ...) where logits is tensor (batch, T)
            # 2) ({task: tensor}, ...)
            # 3) (t1, t2, ...) where each tensor is (batch, 1) or (batch,)
            if len(outputs) >= 1 and isinstance(outputs[0], dict):
                out_dict = outputs[0]
                logits_tensor_list = [
                    out_dict[task].detach().cpu() for task in outcome_list
                ]
                logits = torch.cat(logits_tensor_list, dim=1)

            elif len(outputs) >= 1 and torch.is_tensor(outputs[0]):
                # Assume outputs[0] is logits tensor with shape (batch, T)
                logits = outputs[0].detach().cpu()

            elif all(torch.is_tensor(o) for o in outputs):
                # Tuple of per-task tensors -> concatenate
                logits_tensor_list = [o.detach().cpu() for o in outputs]
                # Ensure each tensor has shape (batch, 1)
                logits_tensor_list = [
                    x.unsqueeze(1) if x.ndim == 1 else x
                    for x in logits_tensor_list
                ]
                logits = torch.cat(logits_tensor_list, dim=1)

            else:
                raise TypeError(
                    f"Unsupported model output structure in tuple/list: "
                    f"{[type(o) for o in outputs]}"
                )

        else:
            # Single tensor output: (batch, T)
            logits = outputs.detach().cpu()

        test_logits.append(logits)
        test_targets.append(y_batch.detach().cpu())

# Aggregate to (N, T) and (N,)
test_logits = np.concatenate(test_logits, axis=0)   # shape (N, T)
test_targets = np.concatenate(test_targets, axis=0) # shape (N, T)     # shape (N,)

calibrator_path = str(CALIBRATOR_PATH)
calibrator = joblib.load(calibrator_path)   # dict: task_name -> IsotonicRegression

# Convert logits to raw probabilities as a fallback
raw_probs = expit(test_logits)              # shape (N, T)

# Container for calibrated probabilities
test_probs = np.zeros_like(test_logits, dtype=float)

for t_idx, task_name in enumerate(outcome_list):
    # Logits for the current task
    scores = test_logits[:, t_idx]

    if task_name in calibrator:
        # Apply the corresponding isotonic regression model
        # IsotonicRegression typically uses transform;
        # some versions may also support predict
        test_probs[:, t_idx] = calibrator[task_name].transform(scores)
    else:
        # If the task was skipped during fitting (e.g., only one class present),
        # fall back to the sigmoid probabilities
        test_probs[:, t_idx] = raw_probs[:, t_idx]

RuntimeError: Error(s) in loading state_dict for DSelectKModelReg:
	Unexpected key(s) in state_dict: "towers.9.fc.weight", "towers.9.fc.bias", "towers.10.fc.weight", "towers.10.fc.bias", "towers.11.fc.weight", "towers.11.fc.bias", "towers.12.fc.weight", "towers.12.fc.bias". 
	size mismatch for shared_layer.S: copying a param with shape torch.Size([13, 13]) from checkpoint, the shape in current model is torch.Size([9, 13]).
	size mismatch for shared_layer.A: copying a param with shape torch.Size([13, 13]) from checkpoint, the shape in current model is torch.Size([9, 13]).
	size mismatch for shared_layer.experts.0.net.0.weight: copying a param with shape torch.Size([256, 62]) from checkpoint, the shape in current model is torch.Size([256, 63]).
	size mismatch for shared_layer.experts.1.net.0.weight: copying a param with shape torch.Size([256, 62]) from checkpoint, the shape in current model is torch.Size([256, 63]).
	size mismatch for shared_layer.experts.2.net.0.weight: copying a param 

In [ ]:
# Run this cell as part of the pipeline.

# Final Evaluation with Confidence Intervals

print("\n=== Test Results with 95% CI (optimal threshold) ===")

macro_auc_list = []
macro_prc_list = []
macro_nprc_list = []

for t_idx, task_name in enumerate(outcome_list):
    y_true = test_targets[:, t_idx]
    y_pred = test_probs[:, t_idx]

    try:
        # AUC / PRC and corresponding confidence intervals
        prev = float(np.mean(y_true))  # prevalence = P(Y=1)

        # AUPRC with CI
        prc_mean, (prc_low, prc_high) = bootstrap_ci(
            average_precision_score, y_true, y_pred
        )

        # Normalized AUPRC relative to random baseline (= prevalence)
        eps = 1e-12
        den = max(prev, eps)

        nprc_mean = prc_mean / den
        nprc_low = prc_low / den
        nprc_high = prc_high / den

        auc_mean, (auc_low, auc_high) = bootstrap_ci(
            roc_auc_score, y_true, y_pred
        )

        macro_auc_list.append(auc_mean)
        macro_prc_list.append(prc_mean)
        macro_nprc_list.append(nprc_mean)

        # Use Youden's J to find optimal threshold and bootstrap Sens/Spec CI
        (sens_mean, (sens_low, sens_high)), \
        (spec_mean, (spec_low, spec_high)), \
        thr_mean = bootstrap_sens_spec_ci(
            y_true, y_pred, n_boot=1000, alpha=0.95
        )

        print(
            f"Task {t_idx:02d} ({task_name}): "
            f"Prev={prev:.4f} | "
            f"nPRC={nprc_mean:.2f} [{nprc_low:.2f}, {nprc_high:.2f}] | "
            f"AUC={auc_mean:.4f} [{auc_low:.4f}, {auc_high:.4f}] | "
            f"PRC={prc_mean:.4f} [{prc_low:.4f}, {prc_high:.4f}] | "
            f"Sens={sens_mean:.3f} [{sens_low:.3f}, {sens_high:.3f}] | "
            f"Spec={spec_mean:.3f} [{spec_low:.3f}, {spec_high:.3f}] | "
            f"Thr={thr_mean:.3f}"
        )

    except ValueError:
        print(
            f"Task {t_idx:02d} ({task_name}): "
            f"AUC/PRC/Sens/Spec could not be calculated"
        )


macro_auc_mean, (macro_auc_low, macro_auc_high) = macro_ci_over_tasks(
    macro_auc_list, n_boot=5000, ci=0.95, seed=42
)
macro_prc_mean, (macro_prc_low, macro_prc_high) = macro_ci_over_tasks(
    macro_prc_list, n_boot=5000, ci=0.95, seed=42
)

print("\n" + "=" * 80)
print(
    f"Macro AUROC: {macro_auc_mean:.4f} "
    f"(95% CI: {macro_auc_low:.4f}, {macro_auc_high:.4f})"
)
print(
    f"Macro AUPRC: {macro_prc_mean:.4f} "
    f"(95% CI: {macro_prc_low:.4f}, {macro_prc_high:.4f})"
)